# Attention QKV Decomposition

Decompose attention through Q, K, V matrices: QK and OV eigenspectra,
positional vs content contributions, value composition, and cross-head alignment.

## Why This Matters

Composition attention qkv decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_qkv_decomposition import (
    qk_eigenspectrum, ov_eigenspectrum,
    positional_vs_content_qk, value_composition_profile,
    cross_head_qkv_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## QK Eigenspectrum

The QK circuit determines what each head attends to. Analyze its eigenspectrum.

In [ ]:
for h in range(4):
    result = qk_eigenspectrum(model, layer=0, head=h)
    print(f"  L0H{h}: eff_rank={result['effective_rank']:.2f}, "
          f"top_eig={result['top_eigenvalue']:.4f}, rank_90={result['rank_90']}")

## OV Eigenspectrum

The OV circuit determines what information is moved. Analyze its singular values.

In [ ]:
for h in range(4):
    result = ov_eigenspectrum(model, layer=0, head=h)
    print(f"  L0H{h}: eff_rank={result['effective_rank']:.2f}, "
          f"top_sv={result['top_singular_value']:.4f}, rank_90={result['rank_90']}")

## Positional vs Content QK

Does the head attend based on position or content?

In [ ]:
for l in range(4):
    for h in range(4):
        result = positional_vs_content_qk(model, tokens, layer=l, head=h)
        pos_type = 'POS' if result['is_positional'] else 'CNT'
        print(f"  L{l}H{h}: pos_corr={result['positional_correlation']:+.3f} [{pos_type}]")

## Value Composition Profile

What information do the value vectors contribute?

In [ ]:
for h in range(4):
    result = value_composition_profile(model, tokens, layer=0, head=h)
    print(f"  L0H{h}: output_norm={result['mean_output_norm']:.4f}, "
          f"value_norm={result['mean_value_norm']:.4f}, diversity={result['value_diversity']:.4f}")

## Cross-Head QKV Alignment

How similar are QKV matrices across heads?

In [ ]:
result = cross_head_qkv_alignment(model, layer=0)
print(f"Mean Q alignment: {result['mean_q_alignment']:.4f}")
print(f"Mean OV alignment: {result['mean_ov_alignment']:.4f}\n")
print('Q alignments:')
for a in result['q_alignments']:
    print(f"  H{a['h1']}-H{a['h2']}: {a['cosine']:+.4f}")
print('\nOV alignments:')
for a in result['ov_alignments']:
    print(f"  H{a['h1']}-H{a['h2']}: {a['cosine']:+.4f}")